# 21 · MCP: build a server and connect a client
Level L4 · Part 4 · core · ~25 min · builds on 07

## Goal
Expose the order tools as an MCP server, connect a client to it over stdio and over Streamable HTTP, and let a model
call a tool through the protocol.

## Why it matters
Lesson 07 gave the model one tool, wired into one script. A second app that needs `lookup_order` (the support chat,
an analyst notebook, a Slack bot) must copy the function, rewrite its schema for its own SDK and keep every copy in
step. Count the work for four apps and four tools:

In [1]:
HOSTS = ["support chat", "analyst notebook", "Slack bot", "refund console"]  # four apps that use the order tools
CAPABILITIES = ["lookup_order", "issue_refund", "open_ticket", "policy text"]  # four things they need from the server

print("adapters without a protocol (each app x each capability):", len(HOSTS) * len(CAPABILITIES))
print("pieces with MCP (each app one client, each capability one server):", len(HOSTS) + len(CAPABILITIES))

adapters without a protocol (each app x each capability): 16
pieces with MCP (each app one client, each capability one server): 8


| Without a protocol | With MCP |
|---|---|
| Each app copies each tool, in its own schema format | Each capability is written once, as an MCP server |
| 4 apps x 4 capabilities = 16 adapters to write and keep in step | 4 clients + 4 servers = 8 pieces, each reused |
| A renamed argument breaks the copy nobody remembered to update | Each client reads the tool list and schema from the server |

**What it makes easy:** one set of tools, many hosts, and the schema lives with the code that runs the tool.
**When you don't need it:** one app, one tool, one team. A plain function (lesson 07) is enough.

## The idea
MCP (the Model Context Protocol) is a standard way for an app (the **host**) to use tools, data and prompt templates
from a **server**. The messages are JSON-RPC (a small standard format: each request is a JSON object with a method
name and parameters; each reply carries the same id). The **client** is the part of the host
that speaks the protocol. The server publishes three kinds of thing: **tools** (actions with a name, a description and an input schema, as in
lesson 07), **resources** (readable data at a URI) and **prompts** (instruction templates). Two **transports** carry
the messages: **stdio**, where the client starts the server as a subprocess on the same machine, and **Streamable
HTTP**, where the server runs at a URL, which is how you deploy it.

Analogy: a USB port. The device states what it can do, and any computer that speaks USB can use it without a driver
written for that one device.

## Picture
```mermaid
flowchart LR
  M[Model] <--> C[MCP client inside the host]
  C -->|stdio: the client starts the server| S[MCP server]
  C -->|Streamable HTTP: a localhost URL| S
  S --> T[tools: lookup_order]
  S --> R[resources: policy text]
  S --> P[prompts: reply template]
```

## Step by step
**Step 1 · the data and the server file.** The server reads the fictional orders and the returns policy from
`data/support_eval.json`, the file lessons 10 to 14 use. It was written by hand for this course and holds no personal
data. Below, a peek at three rows, then the server file is written to `apps/21/`. In the server, `MCPServer` is the SDK's
server class, each decorator publishes one capability, and the type hints and docstring become the schema and the
description the client sees.

In [2]:
import json
import sys
from pathlib import Path

GOLDEN = json.loads(Path("../data/support_eval.json").read_text())
for order_id, row in list(GOLDEN["world"]["orders"].items())[:3]:  # a peek: three of the fictional orders
    print(order_id, row)
print("policy:", GOLDEN["policy"][:70] + "…")
Path("../apps/21").mkdir(parents=True, exist_ok=True)  # the server file goes here

1001 {'status': 'delivered', 'days_late': 0, 'total': 60.0}
1002 {'status': 'in transit', 'days_late': 4, 'total': 45.0}
1003 {'status': 'lost', 'days_late': 9, 'total': 89.5}
policy: Refund only when the order is lost, or delivered and the customer says…


**The package.** This lesson uses the `mcp` Python package, version 2.3.0 or newer, installed with
`pip install "mcp>=2.3.0"` (it is in the course's `requirements.txt`). This lesson uses the v2 names `MCPServer` and
`Client`. Older FastMCP examples use different names (`FastMCP`, `from mcp.server.fastmcp import ...`), so do not mix them.

**The server file.** The cell below writes `apps/21/orders_server.py`. Read it once: it is the whole server.

In [3]:
%%writefile ../apps/21/orders_server.py
"""The Brightside order tools as an MCP server (lesson 21). Run with no argument for stdio; `http <port>` for Streamable HTTP."""
import json
import sys
from pathlib import Path

from mcp.server import MCPServer

GOLDEN = json.loads((Path(__file__).resolve().parents[2] / "data" / "support_eval.json").read_text())
ORDERS = GOLDEN["world"]["orders"]  # fictional rows, the same ones as lessons 10 to 14
POLICY = GOLDEN["policy"]

mcp = MCPServer("brightside-orders", instructions="Read-only order tools for the Brightside Outfitters support desk.",
                log_level="WARNING")


@mcp.tool()
def lookup_order(order_id: str) -> dict:
    """Look up one customer order: its status, days late and total. Use it when the customer gives an order number."""
    if order_id not in ORDERS:  # the error names the next step, as in lesson 07
        return {"error": f"no order {order_id}", "next_step": "Ask for the number in the confirmation email."}
    return ORDERS[order_id]


@mcp.resource("policy://returns")
def returns_policy() -> str:
    """The refund and return policy the support desk follows."""
    return POLICY


@mcp.prompt()
def order_reply(order_id: str) -> str:
    """A ready-made instruction for answering a customer about one order."""
    return (f"Answer the customer about order {order_id}. Call lookup_order first, then reply in two sentences. "
            "Follow the policy resource.")


if __name__ == "__main__":
    if len(sys.argv) == 3 and sys.argv[1] == "http":  # Streamable HTTP on localhost, for a client that uses a URL
        mcp.run("streamable-http", host="127.0.0.1", port=int(sys.argv[2]))
    else:  # stdio: the client starts this file and talks over stdin and stdout
        mcp.run()

Overwriting ../apps/21/orders_server.py


**Step 2 · stdio: the client starts the server.** `StdioServerParameters` names the command to run. The client launches
the server, asks for its tools, and stops it when the `async with` block ends. You start nothing by hand. The
notebook's own Python runs the server, so `sys.executable` is the right command. The client's first request is
`tools/list`, and the server answers with every tool it publishes. The notebook can use `async with` at top level
(top-level await); in a script, wrap it in `asyncio.run(main())`.

In [4]:
from mcp import Client, StdioServerParameters

SERVER = Path("../apps/21/orders_server.py").resolve()
STDIO = StdioServerParameters(command=sys.executable, args=[str(SERVER)])  # how to start the server as a subprocess

async with Client(STDIO) as client:  # top-level await works in a notebook
    tools = (await client.list_tools()).tools
    for tool in tools:
        print(tool.name, "| needs:", tool.input_schema["required"], "|", tool.description)

lookup_order | needs: ['order_id'] | Look up one customer order: its status, days late and total. Use it when the customer gives an order number.


Read the output: the schema came from the server's function, `order_id` is required, and the description is the
docstring. Your client wrote none of it. Next, the same session calls the tool, reads the policy resource and gets the
prompt template.

In [5]:
async with Client(STDIO) as client:
    result = await client.call_tool("lookup_order", {"order_id": "1002"})  # a tool call: runs on the server
    print("is_error:", result.is_error, "|", json.loads(result.content[0].text))
    policy = await client.read_resource("policy://returns")  # a resource: data read by URI
    print("resource:", policy.contents[0].text[:60] + "…")
    prompt = await client.get_prompt("order_reply", {"order_id": "1002"})  # a prompt: a template the host can fill in
    print("prompt:", prompt.messages[0].content.text[:60] + "…")

is_error: False | {'status': 'in transit', 'days_late': 4, 'total': 45.0}
resource: Refund only when the order is lost, or delivered and the cus…
prompt: Answer the customer about order 1002. Call lookup_order firs…


**Step 3 · Streamable HTTP: the same server at a URL.** The server runs in its own process: `http` and a port. Port 8210
is this lesson's first port, so the notebook checks that it is free first. It polls the port until the server
answers, instead of sleeping for a fixed time, then connects by URL. The tool code did not change; only the transport did.
Streamable HTTP is POST-only in the 2026-07-28 spec: each request is one POST to `/mcp`, and the server answers it.

In [6]:
import socket
import subprocess
import time

PORT = 8210  # this lesson's first port (progress.py --ports 21)
URL = f"http://127.0.0.1:{PORT}/mcp"


def port_open(port: int) -> bool:
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0


print("port free before start:", not port_open(PORT))
# If the check below prints False, the server did not start. Run it by hand in a terminal to read its error:
#   python apps/21/orders_server.py http 8210      (from the course folder: python ../apps/21/orders_server.py ...)
# Or send its stderr to a file instead of DEVNULL, e.g. stderr=open("../data/21/server.log", "w").
server = subprocess.Popen([sys.executable, str(SERVER), "http", str(PORT)],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # stderr hidden: server errors do not show here
for _ in range(100):  # poll for up to 10 seconds
    if port_open(PORT) or server.poll() is not None:
        break
    time.sleep(0.1)
print("server answering:", port_open(PORT))

port free before start: True


server answering: True


**Connect by URL.** The same tool, now over Streamable HTTP. Only the client's argument changed: a URL instead of a start command.

In [7]:
async with Client(URL) as client:  # the client connects to a URL, not a command
    result = await client.call_tool("lookup_order", {"order_id": "1003"})
    print("over HTTP:", json.loads(result.content[0].text))

over HTTP: {'status': 'lost', 'days_late': 9, 'total': 89.5}


**Step 4 · a model uses the MCP tools (two model calls).** The client lists the tools and turns each one into a Gemini
function declaration. That is the declaration lesson 07 wrote by hand. Then the loop from lesson 05 runs: the model
asks for a tool, the tool runs on the server, the result goes back, and the model answers. Both model calls are
recorded under `llm_replay` as in lessons 02 to 14, so the replay needs no key. The model is Gemini, the default of the
course.

This step needs a key or the course recording. No key? Skip cells 21 and 23. Nothing later depends on them.
`llm_replay` (records and replays model calls) and `tut_env` (reads the course's `.env` file) are course helpers.
`GOOGLE_CLOUD_API_KEY` is the course's name for the Gemini key. In your own project, drop the two helpers and read
your key with `os.environ`.

In [8]:
sys.path.insert(0, "..")
import llm_replay  # course only: records and replays model calls (not part of the SDK)
llm_replay.start("21_mcp-server-and-client")

from google import genai  # the official Google SDK
from google.genai import types
from tut_env import env  # course only: reads settings from the .env file (not part of the SDK)

REPLAYING = llm_replay.is_replaying()
gemini = genai.Client(vertexai=True, api_key="replay" if REPLAYING else env("GOOGLE_CLOUD_API_KEY"),  # course only
                      http_options=types.HttpOptions(timeout=30_000))
MODEL = "gemini-3.1-flash-lite"  # the course's default model, as in lessons 07 to 14
QUESTION = "Where is my parcel for order 1002? It was due on Monday."
print("client ready · replaying:", REPLAYING)

LLM mode: replay (21_mcp-server-and-client.yaml)


client ready · replaying: True


**The loop.** Each model call is one request. Between the two calls, the client runs the tool the model asked for, on the server.

In [9]:
async with Client(URL) as client:
    tools = (await client.list_tools()).tools
    declarations = [types.FunctionDeclaration(name=t.name, description=t.description, parameters_json_schema=t.input_schema)
                    for t in tools]  # the MCP schema goes straight into the model's tool list
    config = types.GenerateContentConfig(tools=[types.Tool(function_declarations=declarations)], temperature=0)
    history = [types.Content(role="user", parts=[types.Part(text=QUESTION)])]

    first = await gemini.aio.models.generate_content(model=MODEL, contents=history, config=config)  # model call 1
    history.append(first.candidates[0].content)  # the model's turn, unchanged
    responses = []
    for part in first.candidates[0].content.parts:
        if part.function_call:  # the model asked for a tool; the server runs it, not this cell
            # dict(...) turns the SDK's argument map into a plain dict that the MCP call accepts
            outcome = await client.call_tool(part.function_call.name, dict(part.function_call.args))
            # is_error is printed, not handled: the server's error text goes back to the model as the tool result,
            # so the model can read it and answer or ask again. The client needs no special branch for it.
            print("tool over MCP:", part.function_call.name, dict(part.function_call.args), "->", outcome.is_error)
            # the Gemini function response must be a JSON object, so the server's text goes under the key "output"
            responses.append(types.Part.from_function_response(name=part.function_call.name,
                                                               response={"output": outcome.content[0].text}))
    history.append(types.Content(role="user", parts=responses))

    second = await gemini.aio.models.generate_content(model=MODEL, contents=history, config=config)  # model call 2
    print("model answer:", second.text)

tool over MCP: lookup_order {'order_id': '1002'} -> False
model answer: I'm sorry to hear your parcel hasn't arrived yet. According to our records, order 1002 is currently in transit and is 4 days late. I apologize for the delay.


Read the two printed lines. The model asked for `lookup_order` with `order_id` "1002". Your client sent that request to
the server, and the server's answer went back to the model. The model's second call is the final answer. The loop is
the one from lesson 05: only the place the tool runs has changed.

Replay note: inside the recording, the MCP calls of this step are answered from the saved cassette, so the step runs
with no server answer of its own. The live check is the Step 3 call, which runs before the recording starts. The
calls after this point (the mistakes and the Try it) run live against the server. The recording is closed here.

In [10]:
if "llm_replay" in globals():  # the model step may have been skipped (no recording, no key)
    llm_replay.stop()  # saves the recording when one was made (record mode); replay mode just closes it
    print("recording closed:", llm_replay.mode() is None)

recording closed: True


## At work
- Teams put shared tools behind one MCP gateway with a registry. Uber's gateway lists 800+ servers and 5,000+ tools,
  new tools start disabled, and each tool has an access policy → lesson 26 (MCP auth and a gateway) and lesson 31
  (guardrails and audit). Preview, you'll learn this later.
- Stripe's internal MCP server, "Toolshed", holds 400+ tools for its coding agents (Stripe Minions post, 2026-02-09)
  → lesson 26 (what a gateway adds in front of tools). Preview, you'll learn this later.
- The MCP spec made the HTTP core stateless (spec 2026-07-28): any replica can serve any request, but a call lost when
  a replica dies must be sent again. → lesson 34 (serving the fleet) and lesson 35 (durable runs). Preview, you'll
  learn this later.

## Common mistakes
1. **Parsing an error as data.** A wrong argument name does not raise a Python error. The server returns a result with
   `is_error` set, and its text is an error message, not JSON. Parsing that text as JSON fails. Check `is_error` first.
   The cell below does the wrong thing on purpose. This cell is expected to end in a `JSONDecodeError`. The first print
   shows the actual error text the server sent back, so you can see why the parse fails.

In [11]:
async with Client(URL) as client:
    bad = await client.call_tool("lookup_order", {"id": "1002"})  # the wrong argument name
print("is_error:", bad.is_error)
print("server text:", bad.content[0].text)  # the actual error text, shown before the parse below
print(json.loads(bad.content[0].text))  # expected to fail: the text is an error message, not JSON

is_error: True
server text: Error executing tool lookup_order: 1 validation error for lookup_orderArguments
order_id
  Field required [type=missing, input_value={'id': '1002'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

2. **Calling a server that is not running.** The client needs a live server at its URL. With none there, the
   connection fails. The cell below points at a free port (8219, nothing listening) and prints the real error: a
   `ConnectError` with the text "All connection attempts failed". That text does not name the port, so check the URL
   yourself: the host, the port and the path `/mcp`, then start the server. An unknown order is different again: it
   returns an error dict with `is_error` False, as in lesson 07. Check both kinds.

In [12]:
try:
    async with Client("http://127.0.0.1:8219/mcp") as client:  # nothing listens on 8219
        await client.list_tools()
except Exception as err:  # the client wraps the connection error in an ExceptionGroup: print the innermost error
    cause = err
    while getattr(cause, "exceptions", None):
        cause = cause.exceptions[0]
    print("real error:", type(cause).__name__, "|", cause)
print("fix: start the server at this URL, or check the port and the path /mcp")

real error: ConnectError | All connection attempts failed
fix: start the server at this URL, or check the port and the path /mcp


3. **Forgetting `await` on an MCP call.** Every method of `Client` is a coroutine: it runs when you `await` it. Without
   the `await`, the call returns a coroutine object, not a result, and the next line fails. The cell below forgets the
   `await` on purpose. The error is an `AttributeError`, and the fix is the one word.

In [13]:
async with Client(URL) as client:
    pending = client.call_tool("lookup_order", {"order_id": "1002"})  # the await is missing
print("pending is a:", type(pending).__name__)  # a coroutine, not a result
print("is_error:", pending.is_error)  # expected to fail: a coroutine has no is_error

pending is a: coroutine


AttributeError: 'coroutine' object has no attribute 'is_error'

**The fix.** Add the `await`. The call now returns the result, so `is_error` and `content` read as usual.

In [14]:
async with Client(URL) as client:
    done = await client.call_tool("lookup_order", {"order_id": "1002"})  # with the await: a result
print("fix, is_error:", done.is_error, "|", json.loads(done.content[0].text)["status"])

fix, is_error: False | in transit


## Try it
Write `status_of(client, order_id)`: an async function that calls `lookup_order` over HTTP and returns one sentence,
such as "Order 1002: in transit, 4 days late." If the result is an error, return the error text instead. Check both
an order that exists (1002) and one that does not (9999). Hint: an unknown order comes back with `is_error` False.

In [15]:
# your code here
# Scaffold: once status_of is written above, uncomment these two lines and run the cell.
# async with Client(URL) as client:
#     print(await status_of(client, "1002"))

**Answer** — try it first, then expand the hidden cell below.

In [16]:
async def status_of(client, order_id: str) -> str:
    result = await client.call_tool("lookup_order", {"order_id": order_id})
    if result.is_error:
        return result.content[0].text.splitlines()[0]
    data = json.loads(result.content[0].text)
    if "error" in data:  # the server's own error, returned as data with is_error False
        return data["error"] + ". " + data["next_step"]
    return f"Order {order_id}: {data['status']}, {data['days_late']} days late."


async with Client(URL) as client:
    print(await status_of(client, "1002"))
    print(await status_of(client, "9999"))

Order 1002: in transit, 4 days late.
no order 9999. Ask for the number in the confirmation email.


**Clean up.** Stop the HTTP server. The recording was already closed after the model step.

In [17]:
server.terminate()
server.wait(timeout=10)
print("server stopped:", not port_open(PORT))

server stopped: True


## Recap
- MCP lets any host use one server's tools, resources and prompts; the server publishes each schema, so no client
  copies it.
- The same server runs over stdio (the client starts it) or Streamable HTTP (a URL you can deploy); only the transport
  changes.
- A bad argument or an unknown order comes back as a result the model can read, not as a crash.
- The model calls the tools through the client, and the tool runs on the server.

**Interview:** "How would you let three apps use one set of order tools?" Strong answer: "Put the tools behind one
MCP server. Each app gets a client, the schema comes from the server, and the transport is stdio locally or Streamable
HTTP when it is deployed. Then I secure and gate it at the gateway."